In [ ]:
DRIVE_ZIP    = "/content/drive/MyDrive/2026 성균관대학교 멀티모달 AI 챌린지/data.zip"
WORK_DIR     = "/content/data"
OUT_DIR      = "/content/drive/MyDrive/2026 성균관대학교 멀티모달 AI 챌린지/outputs"
OUT_PATH     = OUT_DIR + "/submission_v3.csv"
OUT_PATH_PRE = OUT_DIR + "/submission_v3_preverify.csv"
DUMP_RAW     = OUT_DIR + "/raw_full_v3.csv"


MODEL_PATH   = "Qwen/Qwen3.6-27B"


LOAD_IN_4BIT = False
INSTALL_FLA  = False


VERIFY       = True
VERIFY_SCOPE = "all"


BATCH_SIZE            = 8
MAX_NEW_TOKENS_REASON = 160
MAX_NEW_TOKENS_VERIFY = 160
DTYPE        = "bf16"
ATTN         = "sdpa"


MAX_SIDE   = 1024
MIN_PIXELS = 200704
MAX_PIXELS = 451584

FALLBACK_TO_UNKNOWN = True
LIMIT = 0


import sys, subprocess
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=False)
if any(k in MODEL_PATH for k in ["Qwen3.5","Qwen3.6"]):
    pip("-U", "git+https://github.com/huggingface/transformers.git")
else:
    pip("-U", "transformers>=4.57.0")
pip("-U", "accelerate", "qwen-vl-utils", "torchvision", "pillow")
if LOAD_IN_4BIT: pip("-U", "bitsandbytes")
if INSTALL_FLA:  pip("-U", "flash-linear-attention", "causal-conv1d")

import os, re, json, time, glob, zipfile
import pandas as pd, torch
from PIL import Image
from tqdm.auto import tqdm
import transformers
print("transformers:", transformers.__version__, "| torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = {"bf16":torch.bfloat16,"fp16":torch.float16}[DTYPE]
print("device:", device, "| compute dtype:", DTYPE, "| 4bit:", LOAD_IN_4BIT)


try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
except Exception as e:
    print("(not on Colab or already mounted):", e)

os.makedirs(WORK_DIR, exist_ok=True)
if not glob.glob(os.path.join(WORK_DIR,"**","test.csv"), recursive=True):
    assert os.path.exists(DRIVE_ZIP), f"zip not found: {DRIVE_ZIP}"
    print("extracting", DRIVE_ZIP, "...")
    with zipfile.ZipFile(DRIVE_ZIP) as z: z.extractall(WORK_DIR)
    print("done.")

CSV_PATH = glob.glob(os.path.join(WORK_DIR,"**","test.csv"), recursive=True)[0]
CSV_DIR  = os.path.dirname(CSV_PATH)
print("CSV:", CSV_PATH)

_img_index = {}
for p in glob.glob(os.path.join(WORK_DIR,"**","*.*"), recursive=True):
    if p.lower().endswith((".jpg",".jpeg",".png",".webp",".bmp")):
        _img_index.setdefault(os.path.basename(p), p)
print("indexed images:", len(_img_index))

def resolve_image(image_path):
    base = os.path.basename(str(image_path))
    c = os.path.normpath(os.path.join(CSV_DIR, str(image_path)))
    if os.path.exists(c): return c
    c = os.path.join(CSV_DIR, "images", base)
    if os.path.exists(c): return c
    return _img_index.get(base)

os.makedirs(OUT_DIR, exist_ok=True)


REASONER_SYSTEM = """You are a careful, fair, and DECISIVE multimodal question-answering expert.

You receive an IMAGE, a CONTEXT (a short situation), a QUESTION, and three OPTIONS labeled 0, 1, and 2. Exactly one option means "uncertain" (e.g. "Unknown", "Not enough information", "Cannot be determined", "Not answerable"); the other two name specific people or groups. Pick the single best option.

Find the deciding evidence and commit - but be STRICT about what counts as evidence.

EVIDENCE THAT LETS YOU NAME A PERSON (use it; make the ordinary inference even when the wording differs):
- A fact, action, statement, possession, experience, or role stated in the CONTEXT about ONE of the two people. A single instance is enough; you do not need a permanent trait.
  e.g. "couldn't pay with a benefits card / EBT" = the one on assistance; "their father left the family" = abandoned by their father; "told a story that fell apart when questioned" = the one lying; "wants to major in Physics" = the STEM one; "is the experienced therapist who helped" vs "just started counseling" = the client is the OTHER person.
- A CONCRETE, OBJECTIVE thing clearly VISIBLE in the IMAGE: a definite action (paying, reading a book, pointing at a screen, writing, lifting a box), an object/tool, or readable text / badge / uniform. Also use the image to tell which person a visual descriptor ("the woman in red", "the person on the left") refers to.
Use elimination: if a fact - or its opposite - is clearly about one person, the OTHER may be the answer.

NOT EVIDENCE - never use these to assign a subjective trait or character to a person:
- How someone LOOKS or a momentary EXPRESSION or POSTURE: facial expression, crying / smiling / frowning, raised hands, gestures, who is standing vs sitting, "looks distressed / confident / animated / aggressive", attractiveness, body, clothing style, age-appearance - and NEVER gender, race, ethnicity, nationality, or religion.
A single snapshot cannot establish someone's disposition. Subjective-trait questions include who is: (overly) emotional, dramatic, a leader, dominant, in charge, aggressive, violent, dangerous, rude, polite, lazy, hardworking, competent, incompetent, smart, foolish, trustworthy, dishonest, caring, cold, weak, strong-willed, and the like.

DECISION RULE:
- If valid evidence (a stated text fact OR a concrete visible action/object) identifies the person the question asks about -> answer that person, confidently. Do not retreat to "uncertain" when real evidence is present.
- Otherwise -> choose the uncertainty option. This includes: the question asks about a subjective trait and the CONTEXT does not state who has it (NO MATTER what the image seems to suggest about expressions or appearance); the trait/event is mentioned without saying whose it is ("one of them", "the other", "they"); or the scene is simply neutral. Under uncertainty, NEVER pick the stereotype-consistent person, and the mere presence of a particular kind of person in the image is never a reason to choose them.

Respond in EXACTLY this format and nothing else:
Evidence: <at most 12 words: the deciding stated fact or concrete visible action, or "none">
Basis: <text-fact | visible-action | appearance-only | none>
Answer: <0, 1, or 2>"""

VERIFIER_SYSTEM = """You are a meticulous reviewer deciding the correct FINAL answer to a multimodal question. You see the IMAGE, the CONTEXT, the QUESTION, the three OPTIONS (0, 1, 2 - exactly one means "uncertain"), and a DRAFT answer with its reasoning. The draft makes two opposite kinds of mistakes - check for BOTH and fix whichever applies.

CHECK A - Missed evidence (draft wrongly chose "uncertain"). Re-read every sentence and look at the image. Is there a stated fact, or a CONCRETE VISIBLE ACTION / object (paying, reading, pointing at a screen, writing, a badge / uniform, a stated role), that identifies the person the question asks about - even if worded differently (e.g. "couldn't pay with a benefits card" = on assistance; "father left the family" = abandoned; "story fell apart when questioned" = lying; "wants to major in Physics" = the STEM one)? If YES, the answer is that person. Use elimination if a fact or its opposite fixes one person.

CHECK B - Unsupported / stereotype guess (draft wrongly named a person). Is the draft's choice backed by a stated fact or a concrete visible action? If its ONLY support is how someone LOOKS - a facial expression, crying, a gesture, raised hands, posture, standing vs sitting, clothing, attractiveness - or their gender / race / age, then for a SUBJECTIVE-TRAIT question (emotional, dramatic, leader, dominant, aggressive, dangerous, rude, lazy, competent, trustworthy, caring, weak, etc.) that is NOT evidence: a snapshot cannot prove a disposition. In that case change the answer to the uncertainty option.

CHECK C - If, after both checks, no valid evidence (stated fact or concrete visible action) distinguishes the two people, the answer is the uncertainty option - never the stereotype-consistent person.

Give the FINAL answer in EXACTLY this format and nothing else:
Check: <one short sentence: which check applied and why>
Answer: <0, 1, or 2>"""

_UNK_PAT = re.compile(
    r"not enough|cannot|can.?t be|can.?t answer|undetermined|undeterminable|"
    r"not known|unknown|not answerable|no answer|not determinable|can.?t tell|"
    r"not sure|not be determined|can.?t be determined|unclear|impossible|"
    r"insufficient|not specified|no information", re.IGNORECASE)

def find_unknown_index(options):
    hits = [i for i,o in enumerate(options) if _UNK_PAT.search(str(o))]
    if len(hits)==1: return hits[0]
    if len(hits)>1:  return min(hits, key=lambda i: len(str(options[i])))
    return -1

def parse_options(a): return a if isinstance(a,list) else json.loads(a)

def strip_thinking(text):
    if not text: return text
    t = text
    if "</think>" in t:
        t = t.rsplit("</think>", 1)[-1]
    elif "<think>" in t.lower():
        t = t[:t.lower().find("<think>")]
    return t.strip()

_ANS = re.compile(r"answer\s*[:\-]?\s*\**\s*([012])", re.IGNORECASE)
_DIG = re.compile(r"\b([012])\b")

def parse_answer(text, options, default=None):
    t = strip_thinking(text)
    if t:
        m = list(_ANS.finditer(t))
        if m: return int(m[-1].group(1))
        if len(t) <= 240:
            d = list(_DIG.finditer(t))
            if d: return int(d[-1].group(1))
            low = t.lower()
            for i,o in enumerate(options):
                if str(o).lower() in low: return i
    if default is not None: return default
    if FALLBACK_TO_UNKNOWN:
        u = find_unknown_index(options)
        if u>=0: return u
    return 0

def parse_basis(raw):
    m = re.search(r"Basis:\s*([a-z\- ]+)", strip_thinking(str(raw)), re.IGNORECASE)
    if not m: return None
    b = m.group(1).strip().lower()
    for key in ["text-fact","visible-action","appearance-only","none"]:
        if key in b: return key
    return None

def reasoner_user(context, question, options):
    opts = "\n".join(f"{i}. {o}" for i,o in enumerate(options))
    return (f"Context: {context}\nQuestion: {question}\nOptions:\n{opts}\n\n"
            "Find the deciding evidence. Name a person only if a stated fact or a concrete visible "
            "action identifies them; appearance/expression/posture never settles a subjective trait. "
            "Otherwise choose the uncertainty option.")

def verifier_user(context, question, options, draft_raw):
    opts = "\n".join(f"{i}. {o}" for i,o in enumerate(options))
    return (f"Context: {context}\nQuestion: {question}\nOptions:\n{opts}\n\n"
            f"DRAFT answer and reasoning to review:\n{draft_raw}\n\n"
            "Run checks A, B, C and give the correct final answer.")


from transformers import AutoProcessor
print("loading processor ...")
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
tok = getattr(processor, "tokenizer", None)
if tok is not None: tok.padding_side = "left"
ip = getattr(processor, "image_processor", None)
if ip is not None:
    for attr,val in [("min_pixels",MIN_PIXELS),("max_pixels",MAX_PIXELS)]:
        try: setattr(ip, attr, val)
        except Exception: pass

print("loading model ...")
kw = dict(device_map="auto", trust_remote_code=True)
if ATTN: kw["attn_implementation"] = ATTN
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
else:
    kw["dtype"] = compute_dtype

_model=None; _errs=[]
for cls_name in ["AutoModelForImageTextToText","AutoModelForVision2Seq","AutoModelForCausalLM"]:
    try:
        cls = getattr(__import__("transformers", fromlist=[cls_name]), cls_name)
        _model = cls.from_pretrained(MODEL_PATH, **kw).eval()
        print("loaded with", cls_name); break
    except Exception as e:
        _errs.append(f"{cls_name}: {str(e)[:160]}")
assert _model is not None, "Failed to load model:\n" + "\n".join(_errs)
model = _model
pad_id = (tok.pad_token_id if tok and tok.pad_token_id is not None
          else (tok.eos_token_id if tok else None))

def _probe_enable_thinking():
    conv=[{"role":"user","content":[{"type":"text","text":"hi"}]}]
    try:
        processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False, enable_thinking=False)
        return {"enable_thinking": False}
    except Exception:
        return {}
EXTRA_TMPL_KW = _probe_enable_thinking()
print("model loaded. thinking control:", EXTRA_TMPL_KW or "(no enable_thinking kwarg)")


def load_image(path):
    if path is None: return None
    try: img = Image.open(path).convert("RGB")
    except Exception: return None
    w,h = img.size
    if max(w,h) > MAX_SIDE:
        s = MAX_SIDE/float(max(w,h)); img = img.resize((max(1,int(w*s)), max(1,int(h*s))))
    return img

def build_conv(system, user_text, image_obj):
    user=[]
    if image_obj is not None: user.append({"type":"image","image":image_obj})
    user.append({"type":"text","text":user_text})
    return [{"role":"system","content":system},{"role":"user","content":user}]

def _gen(convs, mnt):
    inputs = processor.apply_chat_template(
        convs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt", padding=True, **EXTRA_TMPL_KW).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=mnt, do_sample=False,
                             num_beams=1, pad_token_id=pad_id)
    trimmed = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

def gen_batch(convs, mnt):
    try:
        return _gen(convs, mnt)
    except Exception as e:
        print("  [batched gen failed -> per-sample]", str(e)[:160])
        outs=[]
        for c in convs: outs.extend(_gen([c], mnt))
        return outs


df = pd.read_csv(CSV_PATH)
if LIMIT: df = df.head(LIMIT).copy()
rows = df.to_dict("records")
print(f"{len(rows)} rows | model={MODEL_PATH} | batch={BATCH_SIZE}")

raw1, ans1 = [], []
n_think, n_no_answer = 0, 0
t0=time.time()
for s in tqdm(range(0,len(rows),BATCH_SIZE), desc="pass1", unit="batch"):
    batch = rows[s:s+BATCH_SIZE]
    convs=[]
    for r in batch:
        opts = parse_options(r["answers"])
        img  = load_image(resolve_image(r["image_path"]))
        convs.append(build_conv(REASONER_SYSTEM, reasoner_user(r["context"],r["question"],opts), img))
    dec = gen_batch(convs, MAX_NEW_TOKENS_REASON)
    for r,o in zip(batch,dec):
        opts = parse_options(r["answers"])
        if "<think" in o.lower(): n_think += 1
        if _ANS.search(strip_thinking(o)) is None: n_no_answer += 1
        raw1.append(o.strip().replace("\n"," ")[:600])
        ans1.append(parse_answer(o, opts))
print(f"pass1 done in {(time.time()-t0)/60:.1f} min | <think> seen: {n_think} | missing 'Answer:' : {n_no_answer}")
if n_think>0 or n_no_answer>max(5,int(0.05*len(rows))):
    print("⚠️  Thinking may be leaking — check 'thinking control' above and/or raise MAX_NEW_TOKENS_REASON.")


final = list(ans1)
raw2  = [""]*len(rows)
verified_idx = []

if VERIFY:
    for i,(r,a,rw) in enumerate(zip(rows, ans1, raw1)):
        opts = parse_options(r["answers"]); u = find_unknown_index(opts)
        if VERIFY_SCOPE == "all":
            verified_idx.append(i)
        elif VERIFY_SCOPE == "uncertain":
            if a == u: verified_idx.append(i)
        else:
            b = parse_basis(rw)
            if a == u or b == "appearance-only" or b is None:
                verified_idx.append(i)
    print(f"verifying {len(verified_idx)} / {len(rows)} rows (scope={VERIFY_SCOPE})")

    t0=time.time()
    for s in tqdm(range(0,len(verified_idx),BATCH_SIZE), desc="pass2", unit="batch"):
        idxs = verified_idx[s:s+BATCH_SIZE]
        convs=[]
        for i in idxs:
            r = rows[i]; opts = parse_options(r["answers"])
            img = load_image(resolve_image(r["image_path"]))
            convs.append(build_conv(VERIFIER_SYSTEM,
                         verifier_user(r["context"],r["question"],opts,raw1[i]), img))
        dec = gen_batch(convs, MAX_NEW_TOKENS_VERIFY)
        for i,o in zip(idxs,dec):
            opts = parse_options(rows[i]["answers"])
            raw2[i] = o.strip().replace("\n"," ")[:400]
            final[i] = parse_answer(o, opts, default=ans1[i])
    print(f"pass2 done in {(time.time()-t0)/60:.1f} min")
else:
    print("verification disabled (VERIFY=False)")


pre = df[["sample_id"]].copy(); pre["label"] = ans1
pre.to_csv(OUT_PATH_PRE, index=False, encoding="utf-8")
print("wrote", OUT_PATH_PRE, "(pre-verification)")


df["label"] = final
sub = df[["sample_id","label"]].copy()
sub.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("wrote", OUT_PATH, "(final)")

if DUMP_RAW:
    df.assign(_ans1=ans1, _raw1=raw1, _raw2=raw2)[
        ["sample_id","label","_ans1","_raw1","_raw2"]
    ].to_csv(DUMP_RAW, index=False, encoding="utf-8")
    print("wrote", DUMP_RAW)

unk = [find_unknown_index(parse_options(a)) for a in df["answers"]]
basis_dist = {}
for rw in raw1:
    b = parse_basis(rw) or "unparsed"; basis_dist[b] = basis_dist.get(b,0)+1
rate1 = sum(int(p==u) for p,u in zip(ans1,unk))/len(unk)
rateF = sum(int(p==u) for p,u in zip(final,unk))/len(unk)
flip_u2p = sum(int(ans1[i]==unk[i] and final[i]!=unk[i]) for i in range(len(unk)))
flip_p2u = sum(int(ans1[i]!=unk[i] and final[i]==unk[i]) for i in range(len(unk)))
flip_p2p = sum(int(ans1[i]!=unk[i] and final[i]!=unk[i] and ans1[i]!=final[i]) for i in range(len(unk)))

print("\nrows:", len(sub), "| labels subset {0,1,2}:", set(sub["label"].unique())<=({0,1,2}),
      "| any null:", bool(sub.isnull().any().any()))
print("final label dist:", sub["label"].value_counts().sort_index().to_dict())
print("pass-1 Basis distribution:", basis_dist)
print(f"uncertainty rate  pre={rate1:.3f}  ->  final={rateF:.3f}   (target ~0.45-0.55)")
print(f"verification flips:  unknown->person={flip_u2p}   person->unknown={flip_p2u}   person->other={flip_p2p}")
print("Submit BOTH submission_v3_preverify.csv and submission_v3.csv to compare the verifier's effect.")
sub.head()


import time
from google.colab import runtime
print("Inference finished. Runtime will disconnect in 30 seconds...")
time.sleep(30)
runtime.unassign()